# 05 — Machine Learning Modeling

## 1. Problem Definition

### Objective
Predict daily store sales using historical Rossmann data.

### Target
- Sales: Daily sales amount for each store.

### Initial Features
- Store
- DayOfWeek
- Promo

### Modeling Strategy
- Start with Linear Regression.
- Train the model using historical observations only.
- Train on open-store observations.
- Predict zero sales for closed stores.
- Evaluate using MAE and RMSE.
- Compare performance against the selected historical mean baseline.

### Important Constraints
- Customers must not be used as a feature because future customer counts are unknown.
- Future Open and Promo values are assumed to be available.
- Use chronological validation to prevent temporal data leakage.

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "train_clean.parquet"

df = pd.read_parquet(DATA_PATH)

print("Dataset shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())

df.head()

In [ ]:
feature_cols = ["Store", "DayOfWeek", "Promo"]
target_col = "Sales"

df[feature_cols + [target_col]].head(10)

## 2. Chronological Data Splitting

Use historical observations before July 2015 for training and July 2015 for validation.

Only open-store observations are used for model fitting and open-day evaluation.

The chronological split prevents future sales targets from entering the training period.

In [ ]:
split_date = pd.Timestamp("2015-07-01")

train_data = df.loc[df["Date"] < split_date].copy()

valid_data = df.loc[df["Date"] >= split_date].copy()

print("Train shape:", train_data.shape)
print("Validation shape:", valid_data.shape)

print("Train max date:", train_data["Date"].max())
print("Validation min date:", valid_data["Date"].min())

In [ ]:
train_open = train_data.loc[train_data["Open"] == 1].copy()
valid_open = valid_data.loc[valid_data["Open"] == 1].copy()

X_train = train_open[feature_cols]
y_train = train_open[target_col]

X_valid = valid_open[feature_cols]
y_valid = valid_open[target_col]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_valid:", X_valid.shape)
print("y_valid:", y_valid.shape)

## 3. Feature Preprocessing

Store and DayOfWeek are categorical features and will be transformed using One-Hot Encoding.

Promo is a binary feature and will be passed through without modification.

The preprocessing transformations must be fitted on training data only to prevent data leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = ["Store", "DayOfWeek"]
binary_features = ["Promo"]


preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("binary", "passthrough", binary_features),
    ],
    sparse_threshold=1.0,
)

In [ ]:
X_train_encoded = preprocessor.fit_transform(X_train)

X_valid_encoded = preprocessor.transform(X_valid)

print("Encoded X_train:", X_train_encoded.shape)
print("Encoded X_valid:", X_valid_encoded.shape)

print("Output type:", type(X_train_encoded).__name__)

## 4. Linear Regression Model

Build an initial Linear Regression model using a Scikit-learn Pipeline.

The pipeline combines categorical feature encoding and model training.

Only open-store observations are used for model fitting.

This initial model assumes additive relationships between the encoded features and sales.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

linear_pipeline = Pipeline(
    steps=[("preprocessor", preprocessor), ("model", LinearRegression())]
)


linear_pipeline.fit(X_train, y_train)


print("Linear Regression training completed.")

In [ ]:
y_pred = linear_pipeline.predict(X_valid)


prediction_preview = pd.DataFrame(
    {"ActualSales": y_valid.to_numpy(), "PredictedSales": y_pred}
)

print("Number of predictions:", len(y_pred))
print("Negative predictions:", (y_pred < 0).sum())


prediction_preview.head(10).round(2)

## 5. Model Evaluation

Evaluate Linear Regression on open-store observations from July 2015.

Use MAE and RMSE to compare the model against the selected historical mean baseline.

Lower values indicate better predictive performance.

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

linear_mae = mean_absolute_error(y_valid, y_pred)
linear_rmse = root_mean_squared_error(y_valid, y_pred)


print(f"Linear Regression MAE: {linear_mae:.2f}")
print(f"Linear Regression RMSE: {linear_rmse:.2f}")

In [ ]:
model_comparison = pd.DataFrame(
    {"Open-day MAE": [732.18, linear_mae], "Open-day RMSE": [1038.49, linear_rmse]}
)

model_comparison.round(2)

### Error Analysis by Promotion Status

Investigate whether Linear Regression prediction errors differ between promotional and non-promotional days.

Compare MAE and mean signed error for both groups.

In [ ]:
linear_error_analysis = valid_open[["Store", "Date", "Promo", "Sales"]].copy()

linear_error_analysis["PredictedSales"] = y_pred

linear_error_analysis["SignedError"] = (
    linear_error_analysis["Sales"] - linear_error_analysis["PredictedSales"]
)


linear_error_analysis["AbsoluteError"] = linear_error_analysis["SignedError"].abs()

linear_error_by_promo = linear_error_analysis.groupby("Promo").agg(
    observation_count=("Sales", "size"),
    mae=("AbsoluteError", "mean"),
    mean_signed_error=("SignedError", "mean"),
)

linear_error_by_promo.round(2)

### Promotion Error Analysis — Findings

Linear Regression achieved an MAE of 926.89 on non-promotional days and 894.31 on promotional days.

Compared with the historical mean baseline, the largest performance gap occurred on non-promotional days.

Mean signed errors were slightly positive and similar across both promotion groups, suggesting limited overall directional bias.

These findings do not support the hypothesis that promotional days are the primary source of Linear Regression's weaker performance.

Further investigation of weekday-level and store-level errors is needed before selecting model improvements.

In [ ]:
linear_error_analysis["DayOfWeek"] = valid_open["DayOfWeek"].to_numpy()

linear_error_by_weekday = linear_error_analysis.groupby("DayOfWeek").agg(
    observation_count=("Sales", "size"),
    mae=("AbsoluteError", "mean"),
    mean_signed_error=("SignedError", "mean"),
)

linear_error_by_weekday.round(2)

## 6. Feature Interaction Experiment

### Hypothesis

Different stores may have different weekday sales patterns.

The initial Linear Regression model assumes a common weekday effect across all stores.

A combined StoreWeekday feature will be introduced to allow store-specific weekday effects.

The new model will be compared against both the initial Linear Regression model and the historical mean baseline.

In [ ]:
X_train_interaction = X_train.copy()
X_valid_interaction = X_valid.copy()

for data in (X_train_interaction, X_valid_interaction):
    data["StoreWeekday"] = (
        data["Store"].astype(str) + "_" + data["DayOfWeek"].astype(str)
    )

interaction_features = ["StoreWeekday", "Promo"]

X_train_interaction[interaction_features].head()

In [ ]:
interaction_preprocessor = ColumnTransformer(
    transformers=[
        (
            "store_weekday",
            OneHotEncoder(handle_unknown="ignore"),
            ["StoreWeekday"],
        ),
        (
            "promo",
            "passthrough",
            ["Promo"],
        ),
    ],
    sparse_threshold=1.0,
)

interaction_pipeline = Pipeline(
    steps=[
        ("preprocessing", interaction_preprocessor),
        ("model", LinearRegression()),
    ]
)

interaction_pipeline.fit(
    X_train_interaction[interaction_features],
    y_train,
)

y_pred_interaction = interaction_pipeline.predict(
    X_valid_interaction[interaction_features]
)

interaction_mae = mean_absolute_error(y_valid, y_pred_interaction)

interaction_rmse = root_mean_squared_error(y_valid, y_pred_interaction)

print(f"Interaction MAE: {interaction_mae:.2f}")
print(f"Interaction RMSE: {interaction_rmse:.2f}")
print("Negative predictions:", (y_pred_interaction < 0).sum())

## 7. Temporal Validation of the Interaction Model

Evaluate the interaction-based Linear Regression model using expanding-window validation.

The model and preprocessing steps are refitted independently for each validation month.

The goal is to determine whether the improvement observed in July generalizes across other time periods.

In [ ]:
from sklearn.base import clone

validation_months = pd.to_datetime(
    [
        "2015-04-01",
        "2015-05-01",
        "2015-06-01",
        "2015-07-01",
    ]
)

interaction_fold_results = []

for start_date in validation_months:
    end_date = start_date + pd.offsets.MonthBegin(1)

    fold_train = df.loc[(df["Date"] < start_date) & (df["Open"] == 1)].copy()

    fold_valid = df.loc[
        (df["Date"] >= start_date) & (df["Date"] < end_date) & (df["Open"] == 1)
    ].copy()

    for data in (fold_train, fold_valid):
        data["StoreWeekday"] = (
            data["Store"].astype(str) + "_" + data["DayOfWeek"].astype(str)
        )

    fold_model = clone(interaction_pipeline)

    fold_model.fit(
        fold_train[interaction_features],
        fold_train["Sales"],
    )

    fold_predictions = fold_model.predict(fold_valid[interaction_features])

    interaction_fold_results.append(
        {
            "month": start_date.strftime("%Y-%m"),
            "open_mae": mean_absolute_error(
                fold_valid["Sales"],
                fold_predictions,
            ),
            "open_rmse": root_mean_squared_error(
                fold_valid["Sales"],
                fold_predictions,
            ),
        }
    )

interaction_cv_results = pd.DataFrame(interaction_fold_results).set_index("month")

interaction_cv_results.round(2)

## 8. Decision Tree Regression

### Objective

Train a Decision Tree Regressor using the same features as the initial Linear Regression model.

Compare the performance against Linear Regression and the historical mean baseline.

The initial tree complexity will be controlled to reduce overfitting.

In [ ]:
from sklearn.base import clone
from sklearn.tree import DecisionTreeRegressor

tree_pipeline = Pipeline(
    steps=[
        ("preprocessing", clone(preprocessor)),
        (
            "model",
            DecisionTreeRegressor(
                max_depth=16,
                min_samples_leaf=50,
                random_state=42,
            ),
        ),
    ]
)

tree_pipeline.fit(X_train, y_train)

tree_predictions = tree_pipeline.predict(X_valid)

tree_mae = mean_absolute_error(
    y_valid,
    tree_predictions,
)

tree_rmse = root_mean_squared_error(
    y_valid,
    tree_predictions,
)

print(f"Decision Tree MAE: {tree_mae:.2f}")
print(f"Decision Tree RMSE: {tree_rmse:.2f}")

In [ ]:
tree_comparison = pd.DataFrame(
    {
        "Open-day MAE": [
            732.18,
            linear_mae,
            interaction_mae,
            tree_mae,
        ],
        "Open-day RMSE": [
            1038.49,
            linear_rmse,
            interaction_rmse,
            tree_rmse,
        ],
    },
    index=[
        "Historical Baseline",
        "Linear Regression",
        "Linear + Interaction",
        "Decision Tree",
    ],
)

tree_comparison.round(2)

### Decision Tree Diagnostics

Compare training and validation errors to investigate whether the model is underfitting or overfitting.

Inspect the actual tree depth and number of leaves to understand the learned model complexity.

In [ ]:
train_tree_predictions = tree_pipeline.predict(X_train)

train_tree_mae = mean_absolute_error(
    y_train,
    train_tree_predictions,
)

train_tree_rmse = root_mean_squared_error(
    y_train,
    train_tree_predictions,
)

tree_diagnostics = pd.DataFrame(
    {
        "MAE": [train_tree_mae, tree_mae],
        "RMSE": [train_tree_rmse, tree_rmse],
    },
    index=["Training", "Validation"],
)

display(tree_diagnostics.round(2))

fitted_tree = tree_pipeline.named_steps["model"]

print("Tree depth:", fitted_tree.get_depth())
print("Number of leaves:", fitted_tree.get_n_leaves())

### Experiment: Increasing Tree Depth

The initial Decision Tree reached its maximum allowed depth of 16
and produced only 140 leaves.

Its high training error suggests possible underfitting.

This experiment increases max_depth from 16 to 32 while keeping
all other settings unchanged.

The objective is to investigate whether model capacity
is limiting predictive performance.

In [ ]:
deeper_tree_pipeline = Pipeline(
    steps=[
        ("preprocessing", clone(preprocessor)),
        (
            "model",
            DecisionTreeRegressor(
                max_depth=32,
                min_samples_leaf=50,
                random_state=42,
            ),
        ),
    ]
)

deeper_tree_pipeline.fit(X_train, y_train)

deep_train_predictions = deeper_tree_pipeline.predict(X_train)
deep_valid_predictions = deeper_tree_pipeline.predict(X_valid)

deep_tree = deeper_tree_pipeline.named_steps["model"]

depth_comparison = pd.DataFrame(
    {
        "Train MAE": [
            train_tree_mae,
            mean_absolute_error(y_train, deep_train_predictions),
        ],
        "Validation MAE": [
            tree_mae,
            mean_absolute_error(y_valid, deep_valid_predictions),
        ],
        "Validation RMSE": [
            tree_rmse,
            root_mean_squared_error(y_valid, deep_valid_predictions),
        ],
        "Tree Depth": [
            fitted_tree.get_depth(),
            deep_tree.get_depth(),
        ],
        "Number of Leaves": [
            fitted_tree.get_n_leaves(),
            deep_tree.get_n_leaves(),
        ],
    },
    index=["Depth 16", "Depth 32"],
)

depth_comparison.round(2)

### Feature Usage Diagnostics

Inspect which encoded features were selected for splitting.

This analysis helps investigate whether the tree can effectively represent store-specific patterns.

In [ ]:
fitted_preprocessor = deeper_tree_pipeline.named_steps["preprocessing"]

deep_tree = deeper_tree_pipeline.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()

split_indices = deep_tree.tree_.feature

split_indices = split_indices[split_indices >= 0]

used_features = pd.Index(feature_names[split_indices]).unique()

store_features_used = sum(
    name.startswith("categorical__Store_") for name in used_features
)

weekday_features_used = sum(
    name.startswith("categorical__DayOfWeek_") for name in used_features
)

print("Total encoded features:", len(feature_names))
print("Distinct features used:", len(used_features))
print("Store indicators used:", store_features_used)
print("Weekday indicators used:", weekday_features_used)
print(
    "Promo used:",
    any("Promo" in name for name in used_features),
)

## 9. CatBoost Regression

### Objective

Evaluate a gradient boosting model with native support for categorical features.

### Hypothesis

CatBoost may learn store-specific patterns more effectively than the previous Decision Tree with One-Hot Encoding.

### Experimental Setup

- Use the same initial features: Store, DayOfWeek, and Promo.
- Keep the July 2015 validation period.
- Train on open-store observations only.
- Compare MAE and RMSE against the historical baseline.

In [ ]:
categorical_features = ["Store", "DayOfWeek"]

X_train_cat = X_train.copy()
X_valid_cat = X_valid.copy()

for column in categorical_features:
    X_train_cat[column] = X_train_cat[column].astype(str)
    X_valid_cat[column] = X_valid_cat[column].astype(str)

X_train_cat.head()

In [ ]:
from catboost import CatBoostRegressor

catboost_model = CatBoostRegressor(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function="RMSE",
    cat_features=categorical_features,
    random_seed=42,
    verbose=False,
)

catboost_model.fit(X_train_cat, y_train)

catboost_predictions = catboost_model.predict(X_valid_cat)

catboost_mae = mean_absolute_error(
    y_valid,
    catboost_predictions,
)

catboost_rmse = root_mean_squared_error(
    y_valid,
    catboost_predictions,
)

print(f"CatBoost MAE: {catboost_mae:.2f}")
print(f"CatBoost RMSE: {catboost_rmse:.2f}")
print("Negative predictions:", (catboost_predictions < 0).sum())

### CatBoost Training Diagnostics

Compare training and validation errors to investigate possible
underfitting, overfitting, or temporal distribution shifts.

Training performance is used for diagnostics, not as an estimate
of generalization performance.

In [ ]:
catboost_train_predictions = catboost_model.predict(X_train_cat)

catboost_train_mae = mean_absolute_error(
    y_train,
    catboost_train_predictions,
)

catboost_train_rmse = root_mean_squared_error(
    y_train,
    catboost_train_predictions,
)

catboost_diagnostics = pd.DataFrame(
    {
        "MAE": [catboost_train_mae, catboost_mae],
        "RMSE": [catboost_train_rmse, catboost_rmse],
    },
    index=["Training", "Validation"],
)

catboost_diagnostics.round(2)

## 10. Calendar Feature Engineering

### Hypothesis

Sales patterns may vary across different months of the year.

Adding Month as a categorical feature may help CatBoost capture seasonal sales patterns.

Month is derived from Date and is available at prediction time.

Only one new feature is introduced to isolate its effect on model performance.

In [ ]:
X_train_calendar = X_train_cat.copy()
X_valid_calendar = X_valid_cat.copy()

X_train_calendar["Month"] = train_open["Date"].dt.month.astype(str)

X_valid_calendar["Month"] = valid_open["Date"].dt.month.astype(str)

X_train_calendar.head()

In [ ]:
calendar_features = ["Store", "DayOfWeek", "Month"]

catboost_calendar_model = CatBoostRegressor(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function="RMSE",
    cat_features=calendar_features,
    random_seed=42,
    verbose=False,
)

catboost_calendar_model.fit(
    X_train_calendar,
    y_train,
)

calendar_predictions = catboost_calendar_model.predict(X_valid_calendar)

calendar_mae = mean_absolute_error(
    y_valid,
    calendar_predictions,
)

calendar_rmse = root_mean_squared_error(
    y_valid,
    calendar_predictions,
)

calendar_comparison = pd.DataFrame(
    {
        "Open-day MAE": [
            732.18,
            catboost_mae,
            calendar_mae,
        ],
        "Open-day RMSE": [
            1038.49,
            catboost_rmse,
            calendar_rmse,
        ],
    },
    index=[
        "Historical Baseline",
        "CatBoost",
        "CatBoost + Month",
    ],
)

calendar_comparison.round(2)

### Temporal Validation of Calendar Features

Compare CatBoost with and without Month using expanding-window validation.

Both models use identical hyperparameters and the same chronological training and validation periods.

The objective is to evaluate whether Month provides consistent improvements across multiple months.

In [ ]:
catboost_fold_results = []

base_features = ["Store", "DayOfWeek", "Promo"]
base_categorical = ["Store", "DayOfWeek"]

for start_date in validation_months:
    end_date = start_date + pd.offsets.MonthBegin(1)

    fold_train = df.loc[(df["Date"] < start_date) & (df["Open"] == 1)].copy()

    fold_valid = df.loc[
        (df["Date"] >= start_date) & (df["Date"] < end_date) & (df["Open"] == 1)
    ].copy()

    for include_month in (False, True):
        X_fold_train = fold_train[base_features].copy()
        X_fold_valid = fold_valid[base_features].copy()

        categorical_cols = base_categorical.copy()

        for data in (X_fold_train, X_fold_valid):
            data[base_categorical] = data[base_categorical].astype(str)

        if include_month:
            X_fold_train["Month"] = fold_train["Date"].dt.month.astype(str)
            X_fold_valid["Month"] = fold_valid["Date"].dt.month.astype(str)

            categorical_cols.append("Month")

        model = CatBoostRegressor(
            iterations=300,
            depth=6,
            learning_rate=0.1,
            loss_function="RMSE",
            cat_features=categorical_cols,
            random_seed=42,
            verbose=False,
        )

        model.fit(X_fold_train, fold_train["Sales"])

        predictions = model.predict(X_fold_valid)

        catboost_fold_results.append(
            {
                "month": start_date.strftime("%Y-%m"),
                "model": ("CatBoost + Month" if include_month else "CatBoost"),
                "mae": mean_absolute_error(
                    fold_valid["Sales"],
                    predictions,
                ),
                "rmse": root_mean_squared_error(
                    fold_valid["Sales"],
                    predictions,
                ),
            }
        )

catboost_cv = pd.DataFrame(catboost_fold_results)

catboost_cv.pivot(
    index="month",
    columns="model",
    values="mae",
).round(2)

## 11. Store-Weekday Interaction in CatBoost

### Hypothesis

Different stores may exhibit different weekday sales patterns.

An explicit StoreWeekday categorical feature may help CatBoost
capture store-specific weekday behavior.

### Experiment

Compare the initial CatBoost model against CatBoost with
StoreWeekday, keeping all hyperparameters unchanged.

In [ ]:
X_train_sw = X_train_cat.copy()
X_valid_sw = X_valid_cat.copy()

for data in (X_train_sw, X_valid_sw):
    data["StoreWeekday"] = data["Store"] + "_" + data["DayOfWeek"]

X_train_sw.head()

In [ ]:
catboost_sw_model = CatBoostRegressor(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function="RMSE",
    cat_features=["Store", "DayOfWeek", "StoreWeekday"],
    random_seed=42,
    verbose=False,
)

catboost_sw_model.fit(X_train_sw, y_train)

sw_predictions = catboost_sw_model.predict(X_valid_sw)

sw_mae = mean_absolute_error(y_valid, sw_predictions)

sw_rmse = root_mean_squared_error(y_valid, sw_predictions)

print(f"CatBoost + StoreWeekday MAE: {sw_mae:.2f}")
print(f"CatBoost + StoreWeekday RMSE: {sw_rmse:.2f}")

### Experiment: SchoolHoliday Feature

Investigate whether school holidays improve CatBoost
sales predictions.

Compare the original CatBoost model with a model that
includes SchoolHoliday, keeping hyperparameters unchanged.

Evaluate both models using July 2015 open-day observations.

In [ ]:
print("SchoolHoliday distribution:")
print(valid_open["SchoolHoliday"].value_counts().sort_index())

print("\nStateHoliday distribution:")
print(valid_open["StateHoliday"].value_counts().sort_index())

In [ ]:
X_train_school = X_train_cat.copy()
X_valid_school = X_valid_cat.copy()

X_train_school["SchoolHoliday"] = train_open["SchoolHoliday"]
X_valid_school["SchoolHoliday"] = valid_open["SchoolHoliday"]

school_model = CatBoostRegressor(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function="RMSE",
    cat_features=["Store", "DayOfWeek"],
    random_seed=42,
    verbose=False,
)

school_model.fit(X_train_school, y_train)

school_predictions = school_model.predict(X_valid_school)

school_mae = mean_absolute_error(y_valid, school_predictions)
school_rmse = root_mean_squared_error(y_valid, school_predictions)

print(f"CatBoost + SchoolHoliday MAE: {school_mae:.2f}")
print(f"CatBoost + SchoolHoliday RMSE: {school_rmse:.2f}")

## 12. Store Metadata Integration

### Objective

Enrich daily sales observations with store-level information.

Before merging, verify that the store metadata contains unique store identifiers and inspect missing values.

Only features available at prediction time should be used for modeling.

In [ ]:
STORE_PATH = PROJECT_ROOT / "data" / "processed" / "store_clean.parquet"

store_df = pd.read_parquet(STORE_PATH)

print("Store metadata shape:", store_df.shape)
print("Unique stores:", store_df["Store"].nunique())
print("Duplicate Store IDs:", store_df["Store"].duplicated().sum())

store_df.head()

In [ ]:
metadata_cols = ["Store", "StoreType", "Assortment"]

train_enriched = train_open.merge(
    store_df[metadata_cols],
    on="Store",
    how="left",
    validate="many_to_one",
)

valid_enriched = valid_open.merge(
    store_df[metadata_cols],
    on="Store",
    how="left",
    validate="many_to_one",
)

print("Train shape:", train_enriched.shape)
print("Validation shape:", valid_enriched.shape)

print("\nMissing store metadata:")
print(train_enriched[["StoreType", "Assortment"]].isna().sum())

train_enriched.head()

In [ ]:
assert len(train_enriched) == len(train_open)
assert len(valid_enriched) == len(valid_open)

assert train_enriched[["StoreType", "Assortment"]].notna().all().all()
assert valid_enriched[["StoreType", "Assortment"]].notna().all().all()

print("Merge validation passed.")

### Experiment: Store Metadata Features

Evaluate whether StoreType and Assortment improve CatBoost predictions.

Keep the original model hyperparameters unchanged to isolate
the effect of adding store-level metadata.

All features must be available at prediction time.

In [ ]:
metadata_features = [
    "Store",
    "DayOfWeek",
    "Promo",
    "StoreType",
    "Assortment",
]

metadata_categorical = [
    "Store",
    "DayOfWeek",
    "StoreType",
    "Assortment",
]

X_train_metadata = train_enriched[metadata_features].copy()
X_valid_metadata = valid_enriched[metadata_features].copy()

for column in metadata_categorical:
    X_train_metadata[column] = X_train_metadata[column].astype(str)
    X_valid_metadata[column] = X_valid_metadata[column].astype(str)

X_train_metadata.head()

In [ ]:
catboost_metadata_model = CatBoostRegressor(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function="RMSE",
    cat_features=metadata_categorical,
    random_seed=42,
    verbose=False,
)

catboost_metadata_model.fit(
    X_train_metadata,
    train_enriched["Sales"],
)

metadata_predictions = catboost_metadata_model.predict(X_valid_metadata)

metadata_mae = mean_absolute_error(
    valid_enriched["Sales"],
    metadata_predictions,
)

metadata_rmse = root_mean_squared_error(
    valid_enriched["Sales"],
    metadata_predictions,
)

print(f"CatBoost + Metadata MAE: {metadata_mae:.2f}")
print(f"CatBoost + Metadata RMSE: {metadata_rmse:.2f}")

### Temporal Validation of Store Metadata

Evaluate CatBoost with StoreType and Assortment across
four expanding-window validation periods.

Compare the results with the original CatBoost model
and the historical mean baseline.

All model parameters remain unchanged.

In [ ]:
all_open_enriched = df.loc[df["Open"] == 1].merge(
    store_df[metadata_cols],
    on="Store",
    how="left",
    validate="many_to_one",
)

assert len(all_open_enriched) == (df["Open"] == 1).sum()

assert all_open_enriched[["StoreType", "Assortment"]].notna().all().all()

print("Enriched data shape:", all_open_enriched.shape)

In [ ]:
metadata_fold_results = []

for start_date in validation_months:
    end_date = start_date + pd.offsets.MonthBegin(1)

    fold_train = all_open_enriched.loc[all_open_enriched["Date"] < start_date]

    fold_valid = all_open_enriched.loc[
        (all_open_enriched["Date"] >= start_date)
        & (all_open_enriched["Date"] < end_date)
    ]

    X_fold_train = fold_train[metadata_features].copy()
    X_fold_valid = fold_valid[metadata_features].copy()

    for column in metadata_categorical:
        X_fold_train[column] = X_fold_train[column].astype(str)
        X_fold_valid[column] = X_fold_valid[column].astype(str)

    fold_model = CatBoostRegressor(
        iterations=300,
        depth=6,
        learning_rate=0.1,
        loss_function="RMSE",
        cat_features=metadata_categorical,
        random_seed=42,
        verbose=False,
    )

    fold_model.fit(
        X_fold_train,
        fold_train["Sales"],
    )

    predictions = fold_model.predict(X_fold_valid)

    metadata_fold_results.append(
        {
            "month": start_date.strftime("%Y-%m"),
            "open_mae": mean_absolute_error(
                fold_valid["Sales"],
                predictions,
            ),
            "open_rmse": root_mean_squared_error(
                fold_valid["Sales"],
                predictions,
            ),
        }
    )

metadata_cv = pd.DataFrame(metadata_fold_results).set_index("month")

metadata_cv.round(2)

## 13. Competition Feature Engineering

### Objective

Investigate whether competition-related information can improve sales predictions.

### Data Validation

Before engineering competition features, inspect missing values,
opening dates, and competition distance distributions.

Historical features must respect the observation date to avoid
using information about competitors that had not yet opened.

In [ ]:
competition_cols = [
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
]

print("Missing values:")
print(store_df[competition_cols].isna().sum())

print("\nCompetition statistics:")
print(store_df[competition_cols].describe().round(2))

print("\nInconsistent opening dates:")
inconsistent_dates = (
    store_df["CompetitionOpenSinceMonth"].isna()
    != store_df["CompetitionOpenSinceYear"].isna()
)

print(inconsistent_dates.sum())

In [ ]:
print("Stores with competition opening year 1900:")

display(
    store_df.loc[
        store_df["CompetitionOpenSinceYear"] == 1900,
        [
            "Store",
            "CompetitionDistance",
            "CompetitionOpenSinceMonth",
            "CompetitionOpenSinceYear",
        ],
    ]
)

print("\nUnknown opening dates:")

unknown_date = store_df["CompetitionOpenSinceYear"].isna()

print(store_df.loc[unknown_date, "CompetitionDistance"].notna().value_counts())

print("\nStores with missing competition distance:")

display(
    store_df.loc[
        store_df["CompetitionDistance"].isna(),
        competition_cols[:],
    ]
)

### Competition Date Features

Create competition-related features using the historical sales date.

- CompetitionDateKnown indicates whether a usable opening date exists.
- CompetitionActive indicates whether the competitor had opened by the observation month.
- CompetitionAgeMonths measures elapsed months since opening.

Unknown opening dates are preserved as missing values.

The year 1900 is treated as uncertain for derived features, without modifying the original metadata.

Competition metadata may originate from a later snapshot, so historical availability remains a modeling limitation.

In [ ]:
all_comp_enriched = all_open_enriched.merge(
    store_df[["Store", *competition_cols]],
    on="Store",
    how="left",
    validate="many_to_one",
)

assert len(all_comp_enriched) == len(all_open_enriched)

comp_year = all_comp_enriched["CompetitionOpenSinceYear"].astype("float64")

comp_month = all_comp_enriched["CompetitionOpenSinceMonth"].astype("float64")

known_date = comp_year.notna() & comp_month.notna() & (comp_year != 1900)

months_since_opening = (
    (all_comp_enriched["Date"].dt.year - comp_year) * 12
    + all_comp_enriched["Date"].dt.month
    - comp_month
)

all_comp_enriched["CompetitionDateKnown"] = known_date.astype("int8")

all_comp_enriched["CompetitionActive"] = (
    (months_since_opening >= 0).astype("float64").where(known_date)
)

all_comp_enriched["CompetitionAgeMonths"] = months_since_opening.clip(lower=0).where(
    known_date
)

In [ ]:
new_competition_features = [
    "CompetitionDateKnown",
    "CompetitionActive",
    "CompetitionAgeMonths",
]

print("Missing values:")
print(all_comp_enriched[new_competition_features].isna().sum())

print("\nCompetition Active distribution:")
print(all_comp_enriched["CompetitionActive"].value_counts(dropna=False))

print("\nCompetition Age statistics:")
print(all_comp_enriched["CompetitionAgeMonths"].describe().round(2))

print("\nNegative competition ages:")
print((all_comp_enriched["CompetitionAgeMonths"] < 0).sum())

### Experiment: Competition Features

Evaluate whether competition-related features improve CatBoost predictions.

The experiment adds CompetitionDistance, CompetitionDateKnown,
CompetitionActive, and CompetitionAgeMonths to the existing
store metadata features.

The original CatBoost hyperparameters remain unchanged.

Missing values are preserved instead of being replaced with zero.

In [ ]:
competition_features = metadata_features + [
    "CompetitionDistance",
    "CompetitionDateKnown",
    "CompetitionActive",
    "CompetitionAgeMonths",
]

comp_train = all_comp_enriched.loc[all_comp_enriched["Date"] < split_date].copy()

comp_valid = all_comp_enriched.loc[all_comp_enriched["Date"] >= split_date].copy()

X_train_comp = comp_train[competition_features].copy()
X_valid_comp = comp_valid[competition_features].copy()

for column in metadata_categorical:
    X_train_comp[column] = X_train_comp[column].astype(str)
    X_valid_comp[column] = X_valid_comp[column].astype(str)

numeric_competition_cols = [
    "CompetitionDistance",
    "CompetitionActive",
    "CompetitionAgeMonths",
]

for column in numeric_competition_cols:
    X_train_comp[column] = X_train_comp[column].astype("float64")
    X_valid_comp[column] = X_valid_comp[column].astype("float64")

print("X_train:", X_train_comp.shape)
print("X_valid:", X_valid_comp.shape)

In [ ]:
catboost_comp_model = CatBoostRegressor(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function="RMSE",
    cat_features=metadata_categorical,
    random_seed=42,
    verbose=False,
)

catboost_comp_model.fit(
    X_train_comp,
    comp_train["Sales"],
)

comp_predictions = catboost_comp_model.predict(X_valid_comp)

comp_mae = mean_absolute_error(
    comp_valid["Sales"],
    comp_predictions,
)

comp_rmse = root_mean_squared_error(
    comp_valid["Sales"],
    comp_predictions,
)

competition_comparison = pd.DataFrame(
    {
        "Open-day MAE": [732.18, metadata_mae, comp_mae],
        "Open-day RMSE": [1038.49, metadata_rmse, comp_rmse],
    },
    index=[
        "Historical Baseline",
        "CatBoost + Metadata",
        "CatBoost + Metadata + Competition",
    ],
)

competition_comparison.round(2)

### Temporal Validation of Competition Features

Evaluate CatBoost with store metadata and competition features
across four expanding-window validation periods.

Compare the results against CatBoost with store metadata only.

Competition features are treated as exploratory because
historical metadata availability is not fully verified.

In [ ]:
competition_fold_results = []

for start_date in validation_months:
    end_date = start_date + pd.offsets.MonthBegin(1)

    fold_train = all_comp_enriched.loc[all_comp_enriched["Date"] < start_date]

    fold_valid = all_comp_enriched.loc[
        (all_comp_enriched["Date"] >= start_date)
        & (all_comp_enriched["Date"] < end_date)
    ]

    X_fold_train = fold_train[competition_features].copy()
    X_fold_valid = fold_valid[competition_features].copy()

    for column in metadata_categorical:
        X_fold_train[column] = X_fold_train[column].astype(str)
        X_fold_valid[column] = X_fold_valid[column].astype(str)

    for column in numeric_competition_cols:
        X_fold_train[column] = X_fold_train[column].astype("float64")
        X_fold_valid[column] = X_fold_valid[column].astype("float64")

    fold_model = CatBoostRegressor(
        iterations=300,
        depth=6,
        learning_rate=0.1,
        loss_function="RMSE",
        cat_features=metadata_categorical,
        random_seed=42,
        verbose=False,
    )

    fold_model.fit(
        X_fold_train,
        fold_train["Sales"],
    )

    predictions = fold_model.predict(X_fold_valid)

    competition_fold_results.append(
        {
            "month": start_date.strftime("%Y-%m"),
            "open_mae": mean_absolute_error(
                fold_valid["Sales"],
                predictions,
            ),
            "open_rmse": root_mean_squared_error(
                fold_valid["Sales"],
                predictions,
            ),
        }
    )

competition_cv = pd.DataFrame(competition_fold_results).set_index("month")

competition_cv.round(2)

### Competition Feature Importance

Inspect the feature importance scores of the trained CatBoost model.

The objective is to understand which features contribute most
to the model's predictions.

Feature importance does not establish causality or prove that
a feature improves generalization performance.

In [ ]:
feature_importance = pd.Series(
    catboost_comp_model.get_feature_importance(),
    index=X_train_comp.columns,
    name="Importance",
)

feature_importance = feature_importance.sort_values(ascending=False)

feature_importance.round(2)

### Ablation Study: CompetitionDistance

Evaluate the contribution of CompetitionDistance by removing it
from the feature set and retraining CatBoost.

All other features, hyperparameters, and validation dates
remain unchanged.

The goal is to measure whether this feature improves predictive
performance, rather than relying only on feature importance.

In [ ]:
features_without_distance = [
    col for col in competition_features if col != "CompetitionDistance"
]

no_distance_model = CatBoostRegressor(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function="RMSE",
    cat_features=metadata_categorical,
    random_seed=42,
    verbose=False,
)

no_distance_model.fit(
    X_train_comp[features_without_distance],
    comp_train["Sales"],
)

no_distance_predictions = no_distance_model.predict(
    X_valid_comp[features_without_distance]
)

no_distance_mae = mean_absolute_error(
    comp_valid["Sales"],
    no_distance_predictions,
)

no_distance_rmse = root_mean_squared_error(
    comp_valid["Sales"],
    no_distance_predictions,
)

ablation_comparison = pd.DataFrame(
    {
        "MAE": [comp_mae, no_distance_mae],
        "RMSE": [comp_rmse, no_distance_rmse],
    },
    index=[
        "With CompetitionDistance",
        "Without CompetitionDistance",
    ],
)

ablation_comparison.round(2)

In [ ]:
no_distance_fold_results = []

for start_date in validation_months:
    end_date = start_date + pd.offsets.MonthBegin(1)

    fold_train = all_comp_enriched.loc[all_comp_enriched["Date"] < start_date]

    fold_valid = all_comp_enriched.loc[
        (all_comp_enriched["Date"] >= start_date)
        & (all_comp_enriched["Date"] < end_date)
    ]

    X_fold_train = fold_train[features_without_distance].copy()
    X_fold_valid = fold_valid[features_without_distance].copy()

    for column in metadata_categorical:
        X_fold_train[column] = X_fold_train[column].astype(str)
        X_fold_valid[column] = X_fold_valid[column].astype(str)

    for column in ["CompetitionActive", "CompetitionAgeMonths"]:
        X_fold_train[column] = X_fold_train[column].astype("float64")
        X_fold_valid[column] = X_fold_valid[column].astype("float64")

    model = CatBoostRegressor(
        iterations=300,
        depth=6,
        learning_rate=0.1,
        loss_function="RMSE",
        cat_features=metadata_categorical,
        random_seed=42,
        verbose=False,
    )

    model.fit(X_fold_train, fold_train["Sales"])

    predictions = model.predict(X_fold_valid)

    no_distance_fold_results.append(
        {
            "month": start_date.strftime("%Y-%m"),
            "open_mae": mean_absolute_error(fold_valid["Sales"], predictions),
        }
    )

no_distance_cv = pd.DataFrame(no_distance_fold_results).set_index("month")

In [ ]:
distance_ablation_cv = pd.DataFrame(
    {
        "With Distance": competition_cv["open_mae"],
        "Without Distance": no_distance_cv["open_mae"],
    }
)

distance_ablation_cv["MAE Increase"] = (
    distance_ablation_cv["Without Distance"] - distance_ablation_cv["With Distance"]
)

distance_ablation_cv.round(2)

### Competition Feature Engineering — Findings

Competition-related features improved CatBoost performance across all four validation months.

The ablation study showed that removing CompetitionDistance increased validation MAE in every fold.

This provides consistent evidence that CompetitionDistance contributes useful predictive information under the current modeling setup.

**Modeling decision:** Retain CompetitionDistance in the experimental feature set.

**Limitation:** Historical availability of competition metadata has not been fully verified. The results remain exploratory and do not establish a causal relationship between competition and sales.

The historical mean baseline remains the strongest model evaluated so far.

## 14. Promo2 Feature Engineering

### Objective

Investigate the Promo2 promotional program and derive
time-aware promotional features.

Before feature engineering, inspect structural missing
values and validate the available start-date information.

In [ ]:
promo2_cols = [
    "Promo2",
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval",
]

print("Promo2 distribution:")
print(store_df["Promo2"].value_counts().sort_index())

print("\nMissing PromoInterval by Promo2:")
print(
    pd.crosstab(
        store_df["Promo2"],
        store_df["PromoInterval"].isna(),
    )
)

print("\nIncomplete records among Promo2 participants:")
promo2_participants = store_df.loc[
    store_df["Promo2"] == 1,
    promo2_cols,
]

print(promo2_participants[promo2_cols[1:]].isna().any(axis=1).sum())

print("\nSample Promo2 participants:")
promo2_participants.head(10)

### Promo2 Start Date Validation

Convert Promo2SinceYear and Promo2SinceWeek into an ISO-week
start date.

For participating stores, the start date is the Monday of the
specified ISO week.

Non-participating stores retain missing start dates.

In [ ]:
promo2_store = store_df[["Store", *promo2_cols]].copy()

is_participant = promo2_store["Promo2"] == 1

iso_week_strings = (
    promo2_store.loc[is_participant, "Promo2SinceYear"].astype("int64").astype(str)
    + "-W"
    + promo2_store.loc[is_participant, "Promo2SinceWeek"]
    .astype("int64")
    .astype(str)
    .str.zfill(2)
    + "-1"
)

promo2_store["Promo2StartDate"] = pd.NaT

promo2_store.loc[is_participant, "Promo2StartDate"] = pd.to_datetime(
    iso_week_strings,
    format="%G-W%V-%u",
    errors="coerce",
)

assert promo2_store.loc[is_participant, "Promo2StartDate"].notna().all()

print("PromoInterval distribution:")
print(promo2_store["PromoInterval"].value_counts(dropna=False))

print("\nSample start dates:")
promo2_store.loc[
    is_participant,
    [
        "Store",
        "Promo2SinceWeek",
        "Promo2SinceYear",
        "Promo2StartDate",
        "PromoInterval",
    ],
].head(10)

### Time-Aware Promo2 Features

Create two features using the sales date and Promo2 metadata:

- Promo2Started: Indicates whether the promotional program has started.
- Promo2Scheduled: Indicates whether the program has started and the observation month belongs to its promotional schedule.

Non-participating stores receive zero for both features.

The features describe the published promotional schedule rather than verified daily promotion activity.

In [ ]:
promo2_metadata = [
    "Store",
    "Promo2",
    "PromoInterval",
    "Promo2StartDate",
]

all_promo2_enriched = all_comp_enriched.merge(
    promo2_store[promo2_metadata],
    on="Store",
    how="left",
    validate="many_to_one",
)

assert len(all_promo2_enriched) == len(all_comp_enriched)

all_promo2_enriched["Promo2StartDate"] = pd.to_datetime(
    all_promo2_enriched["Promo2StartDate"]
)

promo2_started = (all_promo2_enriched["Promo2"] == 1) & (
    all_promo2_enriched["Date"] >= all_promo2_enriched["Promo2StartDate"]
)

month = all_promo2_enriched["Date"].dt.month
interval = all_promo2_enriched["PromoInterval"]

scheduled_month = (
    ((interval == "Jan,Apr,Jul,Oct") & month.isin([1, 4, 7, 10]))
    | ((interval == "Feb,May,Aug,Nov") & month.isin([2, 5, 8, 11]))
    | ((interval == "Mar,Jun,Sept,Dec") & month.isin([3, 6, 9, 12]))
)

all_promo2_enriched["Promo2Started"] = promo2_started.astype("int8")

all_promo2_enriched["Promo2Scheduled"] = (promo2_started & scheduled_month).astype(
    "int8"
)

In [ ]:
new_promo2_features = [
    "Promo2Started",
    "Promo2Scheduled",
]

print("Promo2 feature distributions:")

for column in new_promo2_features:
    print(f"\n{column}")
    print(all_promo2_enriched[column].value_counts())

print("\nConsistency checks:")

assert (
    all_promo2_enriched["Promo2Scheduled"] <= all_promo2_enriched["Promo2Started"]
).all()

assert (
    (
        all_promo2_enriched.loc[
            all_promo2_enriched["Promo2"] == 0,
            new_promo2_features,
        ]
        == 0
    )
    .all()
    .all()
)

before_start = (all_promo2_enriched["Promo2"] == 1) & (
    all_promo2_enriched["Date"] < all_promo2_enriched["Promo2StartDate"]
)

assert (
    (
        all_promo2_enriched.loc[
            before_start,
            new_promo2_features,
        ]
        == 0
    )
    .all()
    .all()
)

print("All Promo2 consistency checks passed.")

### Experiment: Promo2 Features

Evaluate whether Promo2Started and Promo2Scheduled improve
CatBoost sales predictions.

The experiment extends the existing metadata and competition
feature set with two time-aware promotional features.

Model hyperparameters and the July 2015 validation period remain unchanged.
Promo2Scheduled represents the recorded promotional schedule,
not verified daily promotion activity.

In [ ]:
promo2_features = competition_features + [
    "Promo2Started",
    "Promo2Scheduled",
]

promo2_train = all_promo2_enriched.loc[all_promo2_enriched["Date"] < split_date].copy()

promo2_valid = all_promo2_enriched.loc[all_promo2_enriched["Date"] >= split_date].copy()

X_train_promo2 = promo2_train[promo2_features].copy()
X_valid_promo2 = promo2_valid[promo2_features].copy()

for column in metadata_categorical:
    X_train_promo2[column] = X_train_promo2[column].astype(str)
    X_valid_promo2[column] = X_valid_promo2[column].astype(str)

for column in numeric_competition_cols:
    X_train_promo2[column] = X_train_promo2[column].astype("float64")
    X_valid_promo2[column] = X_valid_promo2[column].astype("float64")

print("Train shape:", X_train_promo2.shape)
print("Validation shape:", X_valid_promo2.shape)

In [ ]:
catboost_promo2_model = CatBoostRegressor(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function="RMSE",
    cat_features=metadata_categorical,
    random_seed=42,
    verbose=False,
)

catboost_promo2_model.fit(
    X_train_promo2,
    promo2_train["Sales"],
)

promo2_predictions = catboost_promo2_model.predict(X_valid_promo2)

promo2_mae = mean_absolute_error(
    promo2_valid["Sales"],
    promo2_predictions,
)

promo2_rmse = root_mean_squared_error(
    promo2_valid["Sales"],
    promo2_predictions,
)

promo2_comparison = pd.DataFrame(
    {
        "Open-day MAE": [
            732.18,
            comp_mae,
            promo2_mae,
        ],
        "Open-day RMSE": [
            1038.49,
            comp_rmse,
            promo2_rmse,
        ],
    },
    index=[
        "Historical Baseline",
        "CatBoost + Competition",
        "CatBoost + Competition + Promo2",
    ],
)

promo2_comparison.round(2)

## 15. Final Model Comparison

Compare the main experimental models using July 2015 open-day MAE and RMSE.

The historical baseline results are taken from Notebook 04.

All results in this table refer to development validation, not an independent test set.

In [ ]:
final_model_comparison = pd.DataFrame(
    {
        "Open-day MAE": [
            732.18,
            linear_mae,
            interaction_mae,
            catboost_mae,
            metadata_mae,
            comp_mae,
            promo2_mae,
        ],
        "Open-day RMSE": [
            1038.49,
            linear_rmse,
            interaction_rmse,
            catboost_rmse,
            metadata_rmse,
            comp_rmse,
            promo2_rmse,
        ],
    },
    index=[
        "Historical Baseline",
        "Linear Regression",
        "Linear + Interaction",
        "CatBoost",
        "CatBoost + Metadata",
        "CatBoost + Competition",
        "CatBoost + Competition + Promo2",
    ],
)

final_model_comparison.sort_values("Open-day MAE").round(2)

## 16. Conclusions and Modeling Decisions

### Key Findings

Several machine learning approaches were evaluated for daily Rossmann sales forecasting, including Linear Regression, Decision Tree Regression, and CatBoost.

Feature engineering experiments showed that:

- Store-weekday interactions improved Linear Regression.
- StoreType and Assortment provided consistent but modest improvements to CatBoost.
- Competition-related features produced a larger improvement across the four validation months.
- Removing CompetitionDistance increased validation MAE across all four monthly folds.
- Adding Month provided inconsistent improvements.
- Adding SchoolHoliday or Promo2 features did not improve July validation performance under the tested configurations.

### Model Selection

The historical Store + DayOfWeek + Promo baseline remained the best-performing approach among the evaluated models.

Its July 2015 open-day performance was:

- **MAE: 732.18**
- **RMSE: 1,038.49**

The best experimental CatBoost model used store metadata and competition-related features:

- **MAE: 790.69**
- **RMSE: 1,101.12**

The historical baseline is therefore retained as the current reference model.

### Evaluation Limitations

- The reported July results measure open-store prediction performance.
- Closed-store sales are assumed to be predicted as zero, but the complete evaluation of closed and open stores is handled separately.
- The April–July 2015 validation periods were repeatedly used during model development and cannot be considered an untouched final test set.
- Future Open and Promo values are assumed to be available at prediction time.
- Historical availability of competition metadata has not been fully verified.
- Results do not establish causal relationships between features and sales.

### Final Decision

Retain the historical mean baseline as the current selected model.

Keep CatBoost with metadata and competition features as an experimental alternative.

Future work should focus on reliable evaluation, reproducibility, testing, and deployment rather than increasing model complexity without demonstrated value.